In [ ]:
# ============================================================
# CONFIGURAZIONE AMBIENTE COLAB
# ============================================================

!git clone https://github.com/massafdc/Progetto-AI-VacuumCleaner.git
%cd Progetto-AI-VacuumCleaner

!pip install -r requirements.txt

# Progetto Vacuum-Cleaner

Introduzione all'IA, Informatica 2025/2026<br>
Massa Giorgio <br>
Stamponi Giovanni

# Introduzione
Il progetto Vacuum Cleaner riguarda lo sviluppo di un sistema che simula il comportamento di un robot aspirapolvere che si adatta all'ambiente. <br> L'obiettivo del progetto è integrare tecniche di elaborazione delle immagini, classificatori, algoritmi di ricerca informata e non, al fine di permettere al robot di interpretare l'ambiente e scegliere le azioni più appropriate. <br> Il sistema è composto da diversi moduli, che vengono integrati per ottenere una simulazione completa del comportamento del robot, dall'analisi dell'immagine della foto fino alla pianificazione dei suoi movimenti.

# 1 - Descrizione formale del dominio <br>
**1.1 - Rappresentazione dello stato**

L'ambiente è rappresentato da una griglia quadrata, in cui ogni cella può assumere i seguenti valori: S (Inizio), F (Fine), C (pulita), D (sporca)  V (molto sporca), X (non accessibile).

Gli stati sono rappresentati come S = ( P , G ) dove P rappresenta la posizione corrente del robot e G la configurazione attuale della griglia.

**1.2 Azioni del robot:**

UP/DOWN/LEFT/RIGHT: il robot esegue un movimento verso su/giù/sinistra/destra.
<br>
CLEAN: il robot pulisce la casella su cui si trova.


**1.3 Vincoli e assunzioni:**

*   Nella griglia devono essere presenti solo una cella S ed F ciascuna; queste sono considerate come celle pulite.
* Ogni azione ha costo uguale a 1.
* Il robot si può muovere solo all'interno della tabella: se ad esempio si trova nell'angolo in alto a sinistra, potrà spostarsi solo verso giù o verso destra.
* Quando il robot pulisce una casella D, questa diventa C; se pulisce una casella V, questa diventa D.

**1.4  Goal**
La condizione di goal è che il robot si trovi nella casella F e tutte le celle accessibili abbiano valore C. <br>

# 2 - Ricerca nello spazio degli stati
**2.1 Formulazione come problema di ricerca**

Il dominio è stato implementato come sottoclasse SmartVacuum della classe Problem di AIMA-python, ridefinendo i metodi actions, result, goal_test e path_cost secondo la formalizzazione descritta nel Capitolo 1. Ogni stato è rappresentato come una tupla (posizione, griglia), dove la griglia è a sua volta rappresentata come tupla di tuple, per garantirne l'immutabilità e la possibilità di essere utilizzata come chiave in insiemi e dizionari durante la ricerca

**2.2 Euristica**

Per guidare la ricerca informata (A*) è stata implementata un'euristica basata su una **Minimum Spanning Tree (MST)**, definita come:

$$
h(n) = costo\ minimo\ di\ pulizia\ rimanente + costo\ MST
$$

Il costo minimo di pulizia rimanente viene calcolato sommando 1 per ogni cella `D` ancora presente nella griglia e 2 per ogni cella `V`, poiché una cella `V` richiede due azioni `CLEAN`.

Per il calcolo del costo MST vengono considerati come nodi la posizione corrente del robot, tutte le celle ancora sporche e la posizione finale (goal). Il peso di ogni collegamento tra due nodi è dato dalla relativa **distanza di Manhattan**. La MST rappresenta quindi il costo minimo necessario, in un ambiente privo di ostacoli, per collegare tra loro tutti i punti che devono essere raggiunti.

L'euristica è **ammissibile**, in quanto nessuna delle due componenti sovrastima il costo reale rimanente. Il costo di pulizia rappresenta infatti il numero minimo di azioni `CLEAN` che devono necessariamente essere eseguite. Analogamente, il costo della MST costituisce un limite inferiore sul movimento necessario: la distanza di Manhattan ignora gli ostacoli `X` e quindi può solo sottostimare, o al massimo uguagliare, la distanza effettivamente percorribile dal robot. Inoltre, la MST individua il costo minimo per collegare la posizione del robot, tutte le celle sporche e il goal; qualsiasi soluzione valida deve necessariamente raggiungere tali punti.
Di conseguenza, la somma delle due componenti non può superare il costo della soluzione ottima rimanente:

$$
h(n) \leq h^*(n)
$$

e pertanto l'euristica risulta ammissibile per la ricerca A*.

**2.3 Ricerca informata  non informata**

Per verificare l'efficacia dell'euristica proposta, è stato implementato uno script di confronto (test_search_performance.py) che esegue sulla stessa istanza del problema sia la ricerca in ampiezza (Breadth-First Search, non informata) sia A* (informata), misurando il numero di nodi espansi, il tempo di esecuzione e il costo della soluzione trovata. Per contare i nodi espansi è stato introdotto un contatore (nodes_expanded), incrementato ad ogni chiamata del metodo actions.

In [ ]:
!python /test/test_search_performance.py